## Sensacionalismo

Diferente da ortografia, não existe uma biblioteca pronta que meça
sensacionalismo. A abordagem aqui é por **léxico**: uma lista de
palavras/expressões que costumam aparecer em títulos e textos
sensacionalistas, combinada com dois sinais de formatação (uso de
exclamação e de CAIXA ALTA).

In [2]:
import re

lexico_sensacionalista = [
    "urgente", "bomba", "chocante", "você não vai acreditar",
    "ninguém está falando sobre isso", "revelado", "escândalo",
    "impressionante", "inacreditável", "alerta"
]

def score_sensacionalismo(texto: str) -> dict:
    """
    Recebe o texto de uma noticia e devolve um dicionario com:
      - qtd_lexico: quantas palavras/expressoes da lista sensacionalista apareceram
      - qtd_exclamacoes: quantidade de pontos de exclamacao no texto
      - qtd_caixa_alta: quantidade de palavras inteiramente em CAIXA ALTA
      - qtd_palavras: tamanho do texto em palavras (usado pra normalizar)
      - sinais_por_100_palavras: soma dos tres sinais, normalizada pelo tamanho do texto
    """
    if not isinstance(texto, str) or not texto.strip():
        return {"qtd_lexico": 0, "qtd_exclamacoes": 0, "qtd_caixa_alta": 0,
                "qtd_palavras": 0, "sinais_por_100_palavras": 0.0}

    texto_lower = texto.lower()
    qtd_lexico = sum(texto_lower.count(p) for p in lexico_sensacionalista)
    qtd_exclamacoes = texto.count("!")
    qtd_caixa_alta = len(re.findall(r'\b[A-ZÀ-Ú]{4,}\b', texto))
    qtd_palavras = len(texto.split())

    total_sinais = qtd_lexico + qtd_exclamacoes + qtd_caixa_alta
    sinais_por_100_palavras = (total_sinais / qtd_palavras * 100) if qtd_palavras > 0 else 0.0

    return {
        "qtd_lexico": qtd_lexico,
        "qtd_exclamacoes": qtd_exclamacoes,
        "qtd_caixa_alta": qtd_caixa_alta,
        "qtd_palavras": qtd_palavras,
        "sinais_por_100_palavras": round(sinais_por_100_palavras, 2),
    }

### Por que normalizar aqui também

O mesmo problema da ortografia se repete: uma notícia longa tem mais
chance de acumular exclamações ou palavras em caixa alta só por ter
mais texto, não porque é mais sensacionalista. Por isso somamos os
três sinais e normalizamos pelo tamanho do texto, em vez de comparar
apenas a contagem bruta:

    sinais_por_100_palavras = (qtd_lexico + qtd_exclamacoes + qtd_caixa_alta) / qtd_palavras * 100

**Exemplo:**

| Notícia | Sinais | Palavras | Sinais por 100 palavras |
|---|---|---|---|
| A | 4 | 80 | 5,0 |
| B | 4 | 800 | 0,5 |

As duas notícias têm os mesmos 4 sinais sensacionalistas, mas a
notícia A concentra esses sinais num texto 10 vezes menor — proporcionalmente,
ela é bem mais sensacionalista que a B.

**Um ponto de atenção:** essa lista de palavras é um ponto de partida, não uma
lista definitiva. Vale o grupo revisar e ampliar o `lexico_sensacionalista`
conforme forem testando notícias reais, já que isso é curadoria manual,
não algo que a biblioteca resolve sozinha.

In [3]:
noticias = [
    "Texto da notícia 1 aqui..!!!!!.",
    "Texto da notícia 2 aqui conforme...",
    "Texto da notícia 3 aqui pesquisa...",
]

for i, texto in enumerate(noticias, start=1):
    resultado = score_sensacionalismo(texto)
    print(f"\nNotícia {i}")
    print(f"  Palavras sensacionalistas: {resultado['qtd_lexico']}")
    print(f"  Exclamações: {resultado['qtd_exclamacoes']}")
    print(f"  Palavras em CAIXA ALTA: {resultado['qtd_caixa_alta']}")
    print(f"  Palavras no texto: {resultado['qtd_palavras']}")
    print(f"  Taxa (por 100 palavras): {resultado['sinais_por_100_palavras']}")


Notícia 1
  Palavras sensacionalistas: 0
  Exclamações: 5
  Palavras em CAIXA ALTA: 0
  Palavras no texto: 5
  Taxa (por 100 palavras): 100.0

Notícia 2
  Palavras sensacionalistas: 0
  Exclamações: 0
  Palavras em CAIXA ALTA: 0
  Palavras no texto: 6
  Taxa (por 100 palavras): 0.0

Notícia 3
  Palavras sensacionalistas: 0
  Exclamações: 0
  Palavras em CAIXA ALTA: 0
  Palavras no texto: 6
  Taxa (por 100 palavras): 0.0


## Quantidade de fontes citadas

Aqui usamos duas camadas: a biblioteca `spaCy` para reconhecer nomes de
pessoas e organizações mencionados no texto, e uma lista de verbos de
citação (regex) para contar quantas vezes algo foi atribuído a alguém.

Antes de rodar, é necessário baixar o modelo de português do spaCy uma
única vez:

```bash
python -m spacy download pt_core_news_sm
```

In [ ]:
!python3 -m spacy download pt_core_news_sm

In [4]:
import spacy
import re

nlp = spacy.load("pt_core_news_sm")

verbos_citacao = ["segundo", "de acordo com", "afirmou", "declarou", "disse", "informou", "conforme"]

def contar_fontes(texto: str) -> dict:
    if not isinstance(texto, str) or not texto.strip():
        return {"qtd_entidades": 0, "qtd_verbos_citacao": 0,
                "qtd_palavras": 0, "citacoes_por_100_palavras": 0.0}

    doc = nlp(texto)
    entidades = {ent.text for ent in doc.ents if ent.label_ in ("PER", "ORG")}

    texto_lower = texto.lower()
    qtd_verbos_citacao = sum(texto_lower.count(v) for v in verbos_citacao)

    qtd_palavras = len(texto.split())
    citacoes_por_100_palavras = (qtd_verbos_citacao / qtd_palavras * 100) if qtd_palavras > 0 else 0.0

    return {
        "qtd_entidades": len(entidades),
        "qtd_verbos_citacao": qtd_verbos_citacao,
        "qtd_palavras": qtd_palavras,
        "citacoes_por_100_palavras": round(citacoes_por_100_palavras, 2),
    }

### Entidades x verbos de citação: por que os dois

`qtd_entidades` conta **quantas fontes distintas** aparecem no texto
(nomes de pessoas e organizações) — é uma boa medida de diversidade de
fontes. `qtd_verbos_citacao` conta **quantas vezes algo foi atribuído**
a alguém ("segundo...", "afirmou...") — é uma medida de quanto a notícia
se apoia em declarações, e não só em afirmações soltas do próprio texto.

Uma notícia pode citar 1 fonte só, mas repetidamente ("o ministro disse
X, o ministro também afirmou Y") — nesse caso `qtd_entidades` fica baixo
mas `qtd_verbos_citacao` fica alto. Por isso vale olhar os dois números
juntos na hora de conferir manualmente, em vez de escolher só um.

A normalização por 100 palavras segue a mesma lógica dos critérios
anteriores: notícia longa tende a acumular mais citações só por ter
mais espaço de texto.

In [5]:
for i, texto in enumerate(noticias, start=1):
    resultado = contar_fontes(texto)
    print(f"\nNotícia {i}")
    print(f"  Fontes distintas (entidades): {resultado['qtd_entidades']}")
    print(f"  Verbos de citação: {resultado['qtd_verbos_citacao']}")
    print(f"  Palavras no texto: {resultado['qtd_palavras']}")
    print(f"  Taxa (por 100 palavras): {resultado['citacoes_por_100_palavras']}")


Notícia 1
  Fontes distintas (entidades): 0
  Verbos de citação: 0
  Palavras no texto: 5
  Taxa (por 100 palavras): 0.0

Notícia 2
  Fontes distintas (entidades): 0
  Verbos de citação: 1
  Palavras no texto: 6
  Taxa (por 100 palavras): 16.67

Notícia 3
  Fontes distintas (entidades): 0
  Verbos de citação: 0
  Palavras no texto: 6
  Taxa (por 100 palavras): 0.0


## Estudos prévios

Mesma lógica de léxico do sensacionalismo, com uma lista de palavras-chave
associadas a menções de pesquisa científica ou institucional.

In [6]:
lexico_estudos = [
    "estudo", "pesquisa", "levantamento", "publicado na revista",
    "pesquisadores", "universidade", "instituto", "artigo científico"
]

def contar_mencoes_estudos(texto: str) -> dict:
    """
    Recebe o texto de uma noticia e devolve um dicionario com:
      - qtd_mencoes: quantidade de palavras/expressoes de estudo/pesquisa encontradas
      - qtd_palavras: tamanho do texto em palavras (usado pra normalizar)
      - mencoes_por_100_palavras: qtd_mencoes normalizada pelo tamanho do texto
    """
    if not isinstance(texto, str) or not texto.strip():
        return {"qtd_mencoes": 0, "qtd_palavras": 0, "mencoes_por_100_palavras": 0.0}

    texto_lower = texto.lower()
    qtd_mencoes = sum(texto_lower.count(p) for p in lexico_estudos)
    qtd_palavras = len(texto.split())

    mencoes_por_100_palavras = (qtd_mencoes / qtd_palavras * 100) if qtd_palavras > 0 else 0.0

    return {
        "qtd_mencoes": qtd_mencoes,
        "qtd_palavras": qtd_palavras,
        "mencoes_por_100_palavras": round(mencoes_por_100_palavras, 2),
    }

### Um cuidado com esse critério

A palavra "estudo" ou "universidade" aparecer no texto não garante que
a notícia realmente cita uma pesquisa de verdade — pode ser uma menção
solta ou até um uso enganoso do termo pra parecer mais confiável. Esse
léxico é um primeiro filtro rápido, mas é o tipo de critério onde vale
a pena conferir manualmente algumas notícias antes de confiar no número.

In [7]:
for i, texto in enumerate(noticias, start=1):
    resultado = contar_mencoes_estudos(texto)
    print(f"\nNotícia {i}")
    print(f"  Menções a estudo/pesquisa: {resultado['qtd_mencoes']}")
    print(f"  Palavras no texto: {resultado['qtd_palavras']}")
    print(f"  Taxa (por 100 palavras): {resultado['mencoes_por_100_palavras']}")


Notícia 1
  Menções a estudo/pesquisa: 0
  Palavras no texto: 5
  Taxa (por 100 palavras): 0.0

Notícia 2
  Menções a estudo/pesquisa: 0
  Palavras no texto: 6
  Taxa (por 100 palavras): 0.0

Notícia 3
  Menções a estudo/pesquisa: 1
  Palavras no texto: 6
  Taxa (por 100 palavras): 16.67
